In [1]:
from tensorflow.keras import Model
from tensorflow.keras.layers import Dense, Flatten, Conv2D, BatchNormalization
from tensorflow.keras.optimizers import Adam
from tensorflow_probability import distributions as tfd
import tensorflow as tf
from math import tau, sqrt



class Actor(Model):
    def __init__(self, env, learning_rate=1e-3, **kwargs):
        super().__init__(**kwargs)
        input_shape    = env.observation_space.shape[1:]
        output_shape   = env.action_space.shape[1:]
        self.norm_one  = BatchNormalization()
        self.conv_one  = Conv2D(filters=16, kernel_size=[8, 8], strides=[4, 4],
                                padding='same', activation='relu', name='conv_one')
        self.conv_two  = Conv2D(filters=32, kernel_size=[4, 4], strides=[2, 2],
                                padding='same', activation='relu', name='conv_two')
        self.norm_two  = BatchNormalization()
        self.flatten   = Flatten()
        self.linear    = Dense(256, activation='relu', name='linear')
        self.mu        = Dense(*output_shape, activation=None, name='output_mu')
        self.sigma     = Dense(*output_shape, activation=None, name='output_sigma')
        self.optimizer = Adam(learning_rate=learning_rate)
        self.build((1, *input_shape))

    def call(self, Os):
        Xs = self.norm_one(Os)
        Xs = self.conv_one(Xs)
        Xs = self.conv_two(Xs)
        Xs = self.norm_two(Xs)
        Xs = self.flatten(Xs)
        Xs = self.linear(Xs)
        Mu = self.mu(Xs)
        Sigma = self.sigma(Xs)
        return Mu, Sigma


    def log_prob(self, Os, As):
        Mu, Sigma = self(Os)
        print('Mu', Mu.numpy())
        print('Sigma', Sigma.numpy())
        Ps = 1/(sqrt(tau) * Sigma) * tf.exp(-1/2 * ((As - Mu)/Sigma)**2)
        print(Ps.numpy())
        return tf.reduce_sum(tf.math.log(Ps))
        #N = tfd.MultivariateNormalDiag(loc=Mu, scale_diag=Sigma)
        #return N.log_prob(As)


    def actions(self, Os):
        Mu, Sigma = self.call(Os)
        N  = tfd.MultivariateNormalDiag(loc=Mu, scale_diag=Sigma)
        As = N.sample()
        Ps = N.log_prob(As)
        return As.numpy(), Ps.numpy()


2022-07-07 12:49:55.153169: W tensorflow/stream_executor/platform/default/dso_loader.cc:64] Could not load dynamic library 'libcudart.so.11.0'; dlerror: libcudart.so.11.0: cannot open shared object file: No such file or directory
2022-07-07 12:49:55.153188: I tensorflow/stream_executor/cuda/cudart_stub.cc:29] Ignore above cudart dlerror if you do not have a GPU set up on your machine.


In [2]:
import os
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'
from tqdm import tqdm
import numpy as np
import tensorflow as tf
from math import tau, sqrt
import numpy as np



class Vanilla:
	def __init__(self, env, actor, gamma=0.999, decay=0.9, max_t=None):
		self.env = env
		self.n_envs = env.num_envs
		self.actor = actor
		self.gamma = gamma
		self.decay = decay
		self.max_t = max_t
		self.buffer = [{'O': [], 'R': [], 'A': []} for _ in range(self.n_envs)]


	def __call__(self, episodes):
		Os = self.env.reset()
		self.G_avr = None
		with tqdm(total=episodes) as bar:
			episode = 0
			while episode < episodes:
				As, Ps = self.actor.actions(Os)
				As = np.nan_to_num(As)
				Os, Rs, Ds, _ = self.env.step(As)
				for i in range(self.n_envs):
					self.buffer[i]['O'].append(Os[i])
					self.buffer[i]['A'].append(As[i])
					self.buffer[i]['R'].append(Rs[i])
				for i in np.arange(self.env.num_envs)[Ds]:
					bar.update(1)
					episode += 1
					# REINFORCE UPDATE
					G = sum(self.gamma ** t * R for t, R in enumerate(self.buffer[i]['R']))
					if self.G_avr is None:
						self.G_avr = G
					else:
						self.G_avr = self.G_avr * self.decay + (1 - self.decay) * G
					bar.set_postfix({'G': self.G_avr, 'D': Ds})
					for t, (O, A, R) in enumerate(zip(self.buffer[i]['O'], 
													  self.buffer[i]['A'], 
													  self.buffer[i]['R'])):
						with tf.GradientTape() as tape:
							P = self.actor.log_prob(O[None, ...], A[None, ...])
							target = G * self.gamma**t * P
							gradients = tape.gradient(-target, self.actor.trainable_weights)

						self.actor.optimizer.apply_gradients(zip(gradients, self.actor.trainable_weights))
						print(P.numpy(), target.numpy(), G)
						G = (G - R)/self.gamma
					self.buffer[i] = {'O': [], 'R': [], 'A': []}
		self.env.close()

In [3]:
import gym
from model import Actor
env = gym.vector.make("CarRacing-v1", num_envs=4)


actor = Actor(env)
agent = Vanilla(env, actor)
Os = env.reset()
#agent(128)
#agent.actor.save_weights('vanilla_128.pd')

Track generation: 1021..1280 -> 259-tiles trackTrack generation: 1194..1497 -> 303-tiles trackTrack generation: 1100..1379 -> 279-tiles track
Track generation: 1187..1488 -> 301-tiles track




2022-07-07 12:49:57.505158: W tensorflow/stream_executor/platform/default/dso_loader.cc:64] Could not load dynamic library 'libcuda.so.1'; dlerror: libcuda.so.1: cannot open shared object file: No such file or directory
2022-07-07 12:49:57.505185: W tensorflow/stream_executor/cuda/cuda_driver.cc:269] failed call to cuInit: UNKNOWN ERROR (303)
2022-07-07 12:49:57.505210: I tensorflow/stream_executor/cuda/cuda_diagnostics.cc:156] kernel driver does not appear to be running on this host (typewriter): /proc/driver/nvidia/version does not exist
2022-07-07 12:49:57.505852: I tensorflow/core/platform/cpu_feature_guard.cc:193] This TensorFlow binary is optimized with oneAPI Deep Neural Network Library (oneDNN) to use the following CPU instructions in performance-critical operations:  AVX2 FMA
To enable them in other operations, rebuild TensorFlow with the appropriate compiler flags.


In [16]:
As = env.action_space.sample()
actor.log_prob(Os, As)

[[ 1.8845575e-02  3.1330380e-06 -1.2164510e-02]
 [ 1.9942464e-02  5.7394869e-27 -1.2082870e-02]
 [ 2.4815300e-02  1.1534585e-02 -1.3269917e-02]
 [ 2.6880495e-02  0.0000000e+00 -1.2127635e-02]]


<tf.Tensor: shape=(), dtype=float32, numpy=nan>

In [18]:
Mu, Sigma = actor(Os)
print('Mu', Mu.numpy())
print('Sigma', Sigma.numpy())
print('As', As)
print(((As - Mu)/Sigma)**2)
Ps_log = tf.math.log(1/(sqrt(tau) * tf.abs(Sigma))) + tf.exp(-1/2 * ((As - Mu)/Sigma)**2)
print(Ps_log.numpy())
tf.reduce_sum(Ps_log)

Mu [[ -7.856493   13.435374  -19.321125 ]
 [ -6.5288243  12.018929  -16.405369 ]
 [ -6.739167   10.344818  -17.442099 ]
 [ -5.450582    9.990609  -14.773234 ]]
Sigma [[ 19.416162    2.7355857 -20.401037 ]
 [ 18.745064    1.0955305 -27.681297 ]
 [ 14.865444    4.803723  -19.955502 ]
 [ 13.8821335   0.3600061 -28.469128 ]]
As [[ 0.2161851   0.75203454  0.5572391 ]
 [ 0.23162045  0.07691733  0.03047397]
 [-0.8558698   0.7998415   0.6242406 ]
 [-0.37597877  0.85295844  0.5319229 ]]
tf.Tensor(
[[1.7286552e-01 2.1496447e+01 9.4941652e-01]
 [1.3006987e-01 1.1882450e+02 3.5254216e-01]
 [1.5663393e-01 3.9481514e+00 8.1962466e-01]
 [1.3362622e-01 6.4424255e+02 2.8901958e-01]], shape=(4, 3), dtype=float32)
[[-2.967847   -1.9252627  -3.3124578 ]
 [-2.9128342  -1.0101773  -3.4013047 ]
 [-2.6933064  -2.34944    -3.2486687 ]
 [-2.6141713   0.10269584 -3.4023123 ]]


<tf.Tensor: shape=(), dtype=float32, numpy=-29.735088>